# FFusion 3.5.0 — Version 1 - interactive mode (gradio UX, automatic upload from dagshub)
> Pulls the source image + target video from **DagsHub storage** then runs gradio UI.

Expects:
- target.mp4
- source.jpg

Flow:
- load my custom FF repo
- load the source pic and target vid from dagshub
- run the app in interactive mode (Gradio link)
- make adjustments and run the process
- manually download the video here --OR-- terminate the app session and the output video will be automatically transferred to dagshub for download


**Setup once:** add a Colab Secret named `DAGSHUB_USER_TOKEN` (left sidebar → 🔑) holding a valid DagsHub token.

In [ ]:
# === CELL 1: VERIFY GPU + CLONE THE FORK ===
!nvidia-smi
%cd /content
!rm -rf facefusion
!git clone https://github.com/alt3rhugo/ff.git facefusion
%cd facefusion

In [ ]:
# === CELL 2: INSTALL DEPENDENCIES (latest, via the repo's own installer) ===
!python install.py --onnxruntime cuda --skip-conda

In [ ]:
# === CELL 3: DAGSHUB SYSTEM DEPS + CLIENT ===
!curl -s https://rclone.org/install.sh | sudo bash
!apt-get update -y && apt-get install -y fuse3
%pip install -q "dagshub[jupyter]"

In [ ]:
# === CELL 4: CONFIG + TOKEN (from Colab Secret, no interactive auth) ===
import os
from google.colab import userdata
from datetime import datetime

timestamp = datetime.now().strftime("%H-%M-%S")

os.environ['DAGSHUB_USER_TOKEN'] = userdata.get('DAGSHUB_USER_TOKEN')

REPO_OWNER = 'zbynja'
REPO_NAME = 'ff'
REPO = f'{REPO_OWNER}/{REPO_NAME}'

# Paths inside DagsHub storage (adjust to match what you upload there).
DAGSHUB_SOURCE = 'source.jpg'         # the face to swap in
DAGSHUB_TARGET = 'target.mp4'         # the video to process
DAGSHUB_OUTPUT = f'output/output_{timestamp}.mp4'  # where the result is written back

LOCAL_SOURCE = '/content/input/source.jpg'
LOCAL_TARGET = '/content/input/target.mp4'
LOCAL_FILENAME = f'output_{timestamp}.mp4'
LOCAL_OUTPUT = f'/content/output/{LOCAL_FILENAME}'


In [ ]:
# === CELL 5: DOWNLOAD SOURCE + TARGET FROM DAGSHUB ===
import shutil
import dagshub
import dagshub.storage

mount_path = dagshub.storage.mount(REPO)
print('Mounted at:', mount_path)
print('Root contents:', os.listdir(mount_path))

os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

shutil.copy(os.path.join(mount_path, DAGSHUB_SOURCE), LOCAL_SOURCE)
shutil.copy(os.path.join(mount_path, DAGSHUB_TARGET), LOCAL_TARGET)
print(f'\u2713 source -> {LOCAL_SOURCE}')
print(f'\u2713 target -> {LOCAL_TARGET}')

In [ ]:
#'''
# === CELL 6: Launch the app with Gradio support (interactive mode)
!mkdir -p /content/input
!mkdir -p /content/output
%cd /content/facefusion

!python facefusion.py run \
  --execution-providers cuda \
  --output-path /content/output \
  -s {LOCAL_SOURCE} \
  -t {LOCAL_TARGET} \
  -o {LOCAL_OUTPUT} \
  --processors face_swapper face_enhancer \
  --face-enhancer-model gpen_bfr_512 \
  --face-swapper-model hyperswap_1b_256 \
  --face-swapper-pixel-boost 512x512 \
  --face-enhancer-blend 80 \
  --face-enhancer-weight 0.8 \
  --face-swapper-weight 0.25 \
  --face-mask-blur 0.65 \
  --face-selector-mode one \
  --face-detector-score 0.65 \
  --face-landmarker-score 0.65 \
  --face-detector-angles 0 90 270 \
  --face-mask-types box occlusion

#'''

In [ ]:
import os
from pathlib import Path

def find_file(search_dir, target_filename):
    """
    Search for a file in all subfolders of the given directory.

    Args:
        search_dir (str): The root directory to search in
        target_filename (str): The filename to search for

    Returns:
        list: List of full paths to matching files, or empty list if not found
    """
    matching_files = []

    try:
        # Use rglob to recursively search all subdirectories
        for file_path in Path(search_dir).rglob(target_filename):
            matching_files.append(str(file_path))
    except PermissionError:
        print(f"Permission denied accessing {search_dir}")
    except Exception as e:
        print(f"Error during search: {e}")

    return matching_files

# Usage
if __name__ == "__main__":
    search_directory = "/tmp/gradio"
    target_file = LOCAL_FILENAME

    results = find_file(search_directory, target_file)

    if results:
        print(f"Found {len(results)} file(s):")
        for file_path in results:
            print(f"  {file_path}")
            TEMP_FILE_PATH = file_path
    else:
        print(f"File '{target_file}' not found in {search_directory}")

In [ ]:
LOCAL_OUTPUT = f'{TEMP_FILE_PATH}'

# === CELL 7: UPLOAD RESULT TO DAGSHUB STORAGE BUCKET (S3) ===
# Writes the output into the SAME DagsHub Storage Bucket that CELL 5 reads the
# input from -- not the Git/DVC repo. DagsHub buckets are S3-compatible; the
# DagsHub token serves as both the access key and the secret key.
%pip install -q boto3
import boto3

s3 = boto3.client(
    's3',
    endpoint_url=f'https://dagshub.com/api/v1/repo-buckets/s3/{REPO_OWNER}',
    aws_access_key_id=os.environ['DAGSHUB_USER_TOKEN'],
    aws_secret_access_key=os.environ['DAGSHUB_USER_TOKEN'],
)

# Bucket name == repo name; key == the path inside the bucket.
s3.upload_file(LOCAL_OUTPUT, REPO_NAME, DAGSHUB_OUTPUT)
print(f'✓ uploaded -> s3://{REPO_NAME}/{DAGSHUB_OUTPUT} (DagsHub Storage Bucket)')

In [ ]:
'''
# Disconnect and delete runtime
from google.colab import runtime
runtime.unassign()
'''